<a href="https://colab.research.google.com/github/bagchibijoyini-a11y/hack_test/blob/main/problem1_hack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# STUDENT ACADEMIC EXPERIENCE ANALYZER
# Google Colab - Python Version
# ============================================================

# Install required libraries
!pip -q install transformers torch pandas matplotlib seaborn ipywidgets


# ============================================================
# IMPORT LIBRARIES
# ============================================================

import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import pipeline
from IPython.display import display, HTML
import ipywidgets as widgets


# ============================================================
# LOAD AI SENTIMENT MODEL
# ============================================================

print("Loading AI language model...")

sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

print("AI model loaded successfully!")


# ============================================================
# CONTEXTUAL INDICATOR DICTIONARY
# ============================================================

patterns = {

    "Stress": [
        "stressed",
        "stressful",
        "under stress",
        "feel stressed",
        "feeling stressed",
        "anxious",
        "anxiety",
        "worried",
        "overwhelmed",
        "academic pressure",
        "exam pressure",
        "pressure",
        "cannot cope",
        "can't cope"
    ],

    "Frustration": [
        "frustrated",
        "frustrating",
        "fed up",
        "annoyed",
        "angry",
        "irritated",
        "confused",
        "confusing",
        "struggling",
        "keep failing",
        "nothing works",
        "cannot understand",
        "can't understand"
    ],

    "Cognitive Overload": [
        "too much information",
        "too much to learn",
        "too many concepts",
        "information overload",
        "cognitive overload",
        "cannot concentrate",
        "can't concentrate",
        "difficult to concentrate",
        "hard to concentrate",
        "mentally exhausted",
        "mentally tired",
        "brain is tired",
        "too many subjects",
        "too many topics"
    ],

    "Workload Difficulty": [
        "too much work",
        "heavy workload",
        "large workload",
        "high workload",
        "many assignments",
        "several assignments",
        "multiple assignments",
        "too many assignments",
        "too much homework",
        "many deadlines",
        "tight deadline",
        "deadlines",
        "project workload",
        "assignment workload",
        "backlog",
        "unable to complete",
        "cannot complete",
        "can't complete"
    ],

    "Disengagement": [
        "not interested",
        "no interest",
        "lost interest",
        "don't want to study",
        "do not want to study",
        "don't feel like studying",
        "do not feel like studying",
        "boring",
        "bored",
        "disconnected",
        "disengaged",
        "giving up",
        "give up",
        "skip classes",
        "skipping classes",
        "avoid classes",
        "not motivated",
        "unmotivated"
    ],

    "Positive Experience": [
        "happy",
        "satisfied",
        "enjoy",
        "enjoying",
        "interesting",
        "motivated",
        "motivating",
        "confident",
        "comfortable",
        "manageable",
        "understand well",
        "learning well",
        "doing well",
        "good experience",
        "great experience",
        "helpful",
        "supportive",
        "successful"
    ]
}


# ============================================================
# NEGATION DETECTION
# ============================================================

def is_negated(text, phrase):

    index = text.find(phrase)

    if index == -1:
        return False

    previous_text = text[max(0, index - 50):index]

    negations = [
        "not",
        "no",
        "never",
        "don't",
        "do not",
        "isn't",
        "is not",
        "wasn't",
        "was not",
        "cannot",
        "can't"
    ]

    for word in negations:

        if word in previous_text:
            return True

    return False


# ============================================================
# FIND INDICATORS
# ============================================================

def find_indicators(text):

    text = text.lower()

    results = {}

    for category, phrases in patterns.items():

        detected = []

        for phrase in phrases:

            if phrase in text:

                if not is_negated(text, phrase):

                    detected.append(phrase)

        results[category] = detected

    return results


# ============================================================
# CALCULATE INTENSITY
# ============================================================

def calculate_intensity(text, detected):

    if len(detected) == 0:
        return 0

    score = len(detected) * 20

    strong_words = [
        "very",
        "extremely",
        "severely",
        "completely",
        "constantly",
        "totally"
    ]

    for word in strong_words:

        if word in text.lower():

            score += 15

    return min(score, 100)


# ============================================================
# AI SENTIMENT ANALYSIS
# ============================================================

def get_sentiment(text):

    result = sentiment_model(text[:512])[0]

    label = result["label"]
    confidence = result["score"]

    return label, confidence


# ============================================================
# MAIN ANALYSIS FUNCTION
# ============================================================

def analyze_feedback(text):

    if not text.strip():

        print("Please enter student feedback.")

        return None


    text = text.lower()


    # Contextual analysis
    indicators = find_indicators(text)


    scores = {}

    for category, detected in indicators.items():

        scores[category] = calculate_intensity(
            text,
            detected
        )


    # AI sentiment
    sentiment, confidence = get_sentiment(text)


    # Extract individual scores

    stress = scores["Stress"]

    frustration = scores["Frustration"]

    overload = scores["Cognitive Overload"]

    workload = scores["Workload Difficulty"]

    disengagement = scores["Disengagement"]

    positive = scores["Positive Experience"]


    # ========================================================
    # OVERALL NEGATIVE SCORE
    # ========================================================

    negative_score = np.mean([
        stress,
        frustration,
        overload,
        workload,
        disengagement
    ])


    # ========================================================
    # WELL-BEING SCORE
    # ========================================================

    wellbeing = (
        100
        - negative_score
        + (positive * 0.35)
    )

    wellbeing = int(
        max(
            0,
            min(
                100,
                wellbeing
            )
        )
    )


    # ========================================================
    # DETERMINE DOMINANT STATE
    # ========================================================

    category_scores = {

        "Stress": stress,

        "Frustration": frustration,

        "Cognitive Overload": overload,

        "Workload Difficulty": workload,

        "Disengagement": disengagement,

        "Positive Experience": positive
    }


    dominant = max(
        category_scores,
        key=category_scores.get
    )


    if positive >= 40 and negative_score < 25:

        state = "Positive Academic Experience"
        emoji = "😊"

    elif overload >= 45:

        state = "Cognitive Overload Detected"
        emoji = "🧠"

    elif workload >= 45:

        state = "High Workload Difficulty"
        emoji = "📚"

    elif stress >= 45:

        state = "High Stress Detected"
        emoji = "😟"

    elif frustration >= 45:

        state = "Frustration Detected"
        emoji = "😣"

    elif disengagement >= 45:

        state = "Possible Disengagement"
        emoji = "😴"

    elif negative_score >= 30:

        state = "Academic Difficulty Detected"
        emoji = "⚠️"

    else:

        state = "Generally Stable"
        emoji = "🙂"


    # ========================================================
    # DISPLAY RESULT
    # ========================================================

    print("\n" + "=" * 60)

    print("        STUDENT ACADEMIC EXPERIENCE ANALYSIS")

    print("=" * 60)

    print()

    print(
        f"{emoji} Overall Status : {state}"
    )

    print(
        f"Academic Well-being Score : {wellbeing}/100"
    )

    print(
        f"AI Sentiment : {sentiment}"
    )

    print(
        f"AI Confidence : {confidence:.2%}"
    )

    print()

    print("-" * 60)

    print("CONTEXTUAL INDICATORS")

    print("-" * 60)


    for category, score in scores.items():

        print(
            f"{category:<25} : {score}%"
        )


    # ========================================================
    # DETECTED EVIDENCE
    # ========================================================

    print()

    print("-" * 60)

    print("DETECTED EVIDENCE")

    print("-" * 60)


    found = False


    for category, phrases in indicators.items():

        if phrases:

            found = True

            print(
                f"\n{category}:"
            )

            print(
                "  " + ", ".join(phrases)
            )


    if not found:

        print(
            "No strong contextual indicators detected."
        )


    # ========================================================
    # RECOMMENDATION
    # ========================================================

    print()

    print("-" * 60)

    print("SYSTEM RECOMMENDATION")

    print("-" * 60)


    if stress >= 45:

        print(
            "⚠ Consider academic counselling or stress-support resources."
        )

    elif overload >= 45:

        print(
            "⚠ Consider reducing information load and providing "
            "learning resources in smaller modules."
        )

    elif workload >= 45:

        print(
            "⚠ Consider reviewing assignment deadlines and workload."
        )

    elif disengagement >= 45:

        print(
            "⚠ Consider student engagement activities and academic mentoring."
        )

    elif frustration >= 45:

        print(
            "⚠ Consider additional explanations, examples and faculty support."
        )

    elif positive >= 40:

        print(
            "✓ Student appears to be experiencing a positive academic experience."
        )

    else:

        print(
            "✓ No major academic difficulty indicator detected."
        )


    print()

    print("=" * 60)


    # Return data for dashboard

    return {

        "State": state,

        "Well-being Score": wellbeing,

        "AI Sentiment": sentiment,

        "AI Confidence": confidence,

        "Stress": stress,

        "Frustration": frustration,

        "Cognitive Overload": overload,

        "Workload Difficulty": workload,

        "Disengagement": disengagement,

        "Positive Experience": positive
    }


# ============================================================
# TEST THE SYSTEM
# ============================================================

sample_feedback = """

I have several assignments due this week and
I am finding it difficult to concentrate.
There is too much work and I feel extremely stressed.
I am mentally exhausted and worried about the deadlines.
I am struggling to manage all my subjects.

"""

result = analyze_feedback(sample_feedback)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 23.0 MB/s eta 0:00:00
Loading AI language model...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

AI model loaded successfully!

        STUDENT ACADEMIC EXPERIENCE ANALYSIS

🧠 Overall Status : Cognitive Overload Detected
Academic Well-being Score : 56/100
AI Sentiment : NEGATIVE
AI Confidence : 99.96%

------------------------------------------------------------
CONTEXTUAL INDICATORS
------------------------------------------------------------
Stress                    : 55%
Frustration               : 35%
Cognitive Overload        : 55%
Workload Difficulty       : 75%
Disengagement             : 0%
Positive Experience       : 0%

------------------------------------------------------------
DETECTED EVIDENCE
------------------------------------------------------------

Stress:
  stressed, worried

Frustration:
  struggling

Cognitive Overload:
  difficult to concentrate, mentally exhausted

Workload Difficulty:
  too much work, several assignments, deadlines

------------------------------------------------------------
SYSTEM RECOMMENDATION
----------------------------------------